# Seeded (batch-elimination) audit graphs: Victoria 2025

The 2025 Victorian Senate election is the largest STV election in the
world (65 candidates, 6 seats, N = 4,101,762). A full plausible DFS
graph is computationally intractable at any useful margin, because the
early layers explode combinatorially over the many weak candidates.
**Batch elimination** (writeup §2.3) abridges those layers: it
rigorously justifies eliminating a designated weak set simultaneously,
and seeds the DFS from the surviving configurations.

This notebook builds the seeded graph at the results-table margin
M = 20,000, plots it, and runs both audit drivers. Note the timescales:
construction takes minutes, and each audit pass is dominated by the
sample size, not the graph.


In [ ]:
# Make the repo root importable when the kernel cwd is notebooks/.
import os, sys

repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)


## 1. Load the profile


In [ ]:
from src.election_graphs.numpy_profile import load_numpy

profile, m, *_ = load_numpy(
    os.path.join(repo_root, "data", "australia_federal", "vic_2025_votekit.csv")
)
print(f"C = {len(profile.candidates)} candidates, m = {m} seats, "
      f"N = {round(profile.total_ballot_wt)} ballots")


## 2. Seeded construction

`seeded_build` partitions the candidates into very-strong / strong / weak
sets from their maximum possible tallies (it prints the partition), seats
any forced very-strong winners, allocates one seed vertex per plausible
configuration of the strong candidates, and only then runs the DFS.
The resulting graph is a few hundred vertices instead of an intractable
full DFS — and audits of it target the same reported outcome at the same
margin, so nothing is lost.


In [ ]:
from src.wigm_graphs.seeded import SeededWIGMGraphConstructor

constructor = SeededWIGMGraphConstructor(
    profile,
    m=m,
    MoI=20_000,
    memory_lite=True,
    simultaneous=True,
)
constructor.seeded_build()
constructor.add_natural_edges()
constructor.assign_tightest_margins()
constructor.coherence_check()


## 3. Plot it

The black connector on the left marks the pre-seated very-strong
winners; each seed vertex continues one plausible configuration of the
strong candidates.


In [ ]:
from src.plotting import plot_wigm_graph

plot_wigm_graph(
    constructor,
    figsize=(20, 8),
    node_size=100,
    font_size=14,
    label_edges="minimal",
    label_vertices=True,
    plot_horizontal=True,
    title="Victoria 2025 (C=65, m=6, N=4,101,762, M=20,000)",
)


## 4. Delta-method audit

One seeded trial at the certifying sample size from the results table
(0.5% of ballots, 20,508 of them). This is the audit that makes
Victoria's row: the Delta method certifies at half a percent of the
ballots cast.


In [ ]:
from src.test_processes.delta_method import DeltaMethodAuditDriver

delta_driver = DeltaMethodAuditDriver(
    constructor,
    fractional_sample_size=1 / 200,
    noise_level=0.02,
    seed=0,
    alpha=0.05,
    verbose=True,
)
success = delta_driver.run()
print(f"certified: {success} with a sample of {delta_driver.sample_size} ballots")


## 5. Mismatch-based audit

At the 2% artificial noise level of the results table, the mismatch
audit does **not** certify Victoria (its ASN column is an X): the
diluted margin M/N = 0.5% is small relative to the mismatch rate. To
demonstrate the driver without sampling to the N/2 cap, we bound the
sample at 20,000 ballots and watch how many escape-edge compilers
certify.


In [ ]:
from src.test_processes.driver import GlobalAuditDriver

mismatch_driver = GlobalAuditDriver(
    constructor,
    noise_level=0.02,
    sample_size=20_000,
    seed=0,
    compiler_type="noise",
    simultaneous=True,
    alpha=0.05,
    print_diagnostics_every=2_000,
)
print(f"escape-edge compilers: {len(mismatch_driver.compilers)}")

certified = mismatch_driver.run(constructor.ballot_matrix)
print(
    f"certified: {certified} — "
    f"{len(mismatch_driver.certified)}/{len(mismatch_driver.compilers)} "
    f"compilers passed after {mismatch_driver.i} ballots"
)


At realistic real-world mismatch rates (typically below 0.2%) the
mismatch audit performs far better — the 2% level here follows the
deliberately adversarial protocol of §3.4.
